
# AstroRAG — Data Ingestion Pipeline

Este notebook faz parte do projeto **AstroRAG**, uma plataforma GenAI para assistência científica em radioastronomia utilizando:
- RAG (Retrieval-Augmented Generation)
- LLMs open-source
- Embeddings vetoriais
- Agentes inteligentes

## Objetivo deste notebook

Neste primeiro módulo iremos:

1. Carregar PDFs científicos;
2. Extrair texto;
3. Dividir o conteúdo em chunks;
4. Preparar os dados para embeddings e RAG.



# Instalação das bibliotecas

Execute a célula abaixo caso ainda não tenha instalado as dependências.


In [1]:
# Execute apenas se necessário

!pip install langchain 
!pip install langchain-community 
!pip install pypdf 
!pip install pymupdf 
!pip install sentence-transformers 
!pip install faiss-cpu 



# Imports


In [13]:

import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter



# Estrutura de diretórios

Criando uma estrutura simples para armazenar os PDFs.


In [14]:

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Diretório criado: {RAW_DIR}")


Diretório criado: /Users/rafas/anaconda_projects/portifolio/AstroRag/data/raw



# Adicionar PDFs científicos

Coloque os artigos científicos em:

```text
data/raw/
```

Sugestões:
- papers sobre FRBs;
- artigos do CHIME;
- radioastronomia;
- classificação de sinais.



# Listar PDFs disponíveis


In [15]:
pdf_files = list(RAW_DIR.glob("*.pdf"))

print(f"Quantidade de PDFs encontrados: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Quantidade de PDFs encontrados: 6
2108.00559v3.pdf
2511.02328v1.pdf
PinchukMargot22.aj163.SETI_ML.pdf
1803.11235v1.pdf
2212.03972v2.pdf
new_frb.pdf



# Carregar PDFs

Utilizaremos o `PyPDFLoader` do LangChain.


In [16]:
documents = []

for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    documents.extend(docs)

print(f"Total de páginas carregadas: {len(documents)}")

Total de páginas carregadas: 189



# Visualizar conteúdo carregado


In [17]:
if len(documents) > 0:
    print(documents[0].page_content[:2000])
else:
    print("Nenhum documento carregado.")

Draft version December 24, 2021
Typeset using LATEX default style in AASTeX63
A Machine Learning-based Direction-of-origin Filter for the Identiﬁcation of Radio Frequency
Interference in the Search for Technosignatures
Pavlo Pinchuk1 and Jean-Luc Margot 2, 1
1Department of Physics and Astronomy, University of California, Los Angeles, CA 90095, USA
2Department of Earth, Planetary, and Space Sciences, University of California, Los Angeles, CA 90095, USA
(Received July 28, 2021; Revised October 25, 2021; Accepted December 7, 2021)
Submitted to Astronomical Journal
ABSTRACT
Radio frequency interference (RFI) mitigation remains a major challenge in the search for radio
technosignatures. Typical mitigation strategies include a direction-of-origin (DoO) ﬁlter, where a signal
is classiﬁed as RFI if it is detected in multiple directions on the sky. These classiﬁcations generally
rely on estimates of signal properties, such as frequency and frequency drift rate. Convolutional neural
networks (CN


# Divisão em chunks

Chunking é essencial para sistemas RAG.

## Parâmetros

- `chunk_size`: tamanho do trecho;
- `chunk_overlap`: sobreposição entre chunks.


In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Quantidade total de chunks: {len(chunks)}")

Quantidade total de chunks: 1763



# Visualizar chunks


In [19]:
if len(chunks) > 0:
    print(chunks[0].page_content)
else:
    print("Nenhum chunk criado.")

Draft version December 24, 2021
Typeset using LATEX default style in AASTeX63
A Machine Learning-based Direction-of-origin Filter for the Identiﬁcation of Radio Frequency
Interference in the Search for Technosignatures
Pavlo Pinchuk1 and Jean-Luc Margot 2, 1
1Department of Physics and Astronomy, University of California, Los Angeles, CA 90095, USA
2Department of Earth, Planetary, and Space Sciences, University of California, Los Angeles, CA 90095, USA



# Estatísticas simples


In [20]:
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

if len(chunk_lengths) > 0:
    print(f"Menor chunk: {min(chunk_lengths)}")
    print(f"Maior chunk: {max(chunk_lengths)}")
    print(f"Média: {sum(chunk_lengths)/len(chunk_lengths):.2f}")
else:
    print("Nenhum chunk disponível.")

Menor chunk: 15
Maior chunk: 500
Média: 447.93



# Próximos passos

No próximo notebook iremos:

- gerar embeddings;
- criar banco vetorial FAISS;
- realizar busca semântica;
- construir o pipeline RAG.


In [21]:
import pickle

with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Chunks salvos com sucesso!")

Chunks salvos com sucesso!
